<a href="https://colab.research.google.com/github/uvindu827/J26-DS-350/blob/Indhi_Component2_CHL_Forecast/03_XGBoost_Final_Productivity_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ======================================================================
# STEP 31.9P-16R
# FINAL TUNED XGBOOST PRODUCTIVITY CLASSIFICATION
# ======================================================================

import os
import json
import pandas as pd
import numpy as np

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("=" * 70)
print("STEP 31.9P-16R")
print("FINAL TUNED XGBOOST PRODUCTIVITY CLASSIFICATION")
print("=" * 70)


# ======================================================================
# PROJECT PATHS
# ======================================================================

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

TUNED_FINAL_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization",
    "final_tuned_candidate"
)

TEST_PREDICTIONS_FILE = os.path.join(
    TUNED_FINAL_DIR,
    "final_test_evaluation",
    "xgboost_final_tuned_test_predictions.csv"
)

PRODUCTIVITY_THRESHOLD_FILE = os.path.join(
    PROJECT_DIR,
    "06_models",
    "18_productivity_classification",
    "productivity_thresholds_training_only.csv"
)

OUTPUT_DIR = os.path.join(
    TUNED_FINAL_DIR,
    "productivity_classification"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print(f"\nProject:")
print(PROJECT_DIR)

print("\nFinal tuned XGBoost directory:")
print(TUNED_FINAL_DIR)

print("\nTuned XGBoost test predictions:")
print(TEST_PREDICTIONS_FILE)

print("\nTraining-only productivity thresholds:")
print(PRODUCTIVITY_THRESHOLD_FILE)

print("\nOutput directory:")
print(OUTPUT_DIR)


# ======================================================================
# REQUIRED FILE CHECK
# ======================================================================

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

print(
    f"\nTuned XGBoost test predictions: "
    f"{os.path.exists(TEST_PREDICTIONS_FILE)}"
)

print(
    f"Training-only threshold file: "
    f"{os.path.exists(PRODUCTIVITY_THRESHOLD_FILE)}"
)

if not os.path.exists(TEST_PREDICTIONS_FILE):
    raise FileNotFoundError(
        "Final tuned XGBoost test predictions were not found."
    )

if not os.path.exists(PRODUCTIVITY_THRESHOLD_FILE):
    raise FileNotFoundError(
        "Training-only productivity threshold file was not found."
    )

print("\n✓ All required files found.")


# ======================================================================
# METHODOLOGICAL SAFETY
# ======================================================================

print("\n" + "=" * 70)
print("METHODOLOGICAL SAFETY")
print("=" * 70)

print("""
Training data:
    NOT LOADED

Validation data:
    NOT LOADED

Raw test dataset:
    NOT LOADED

XGBoost model:
    NOT LOADED

Existing tuned XGBoost test predictions:
    ALREADY GENERATED
    WILL ONLY BE READ

Productivity thresholds:
    TRAINING-DERIVED
    WILL ONLY BE READ

XGBoost retraining:
    NOT PERFORMED

Hyperparameter modification:
    NOT PERFORMED

Prediction regeneration:
    NOT PERFORMED

Threshold recalculation:
    NOT PERFORMED
""")

print("✓ Only saved final tuned predictions will be classified.")
print("✓ Training-only thresholds remain locked.")


# ======================================================================
# LOAD TRAINING-ONLY THRESHOLDS
# ======================================================================

print("\n" + "=" * 70)
print("LOADING TRAINING-ONLY PRODUCTIVITY THRESHOLDS")
print("=" * 70)

threshold_df = pd.read_csv(
    PRODUCTIVITY_THRESHOLD_FILE
)

print("\nThreshold file:")
print(threshold_df.to_string(index=False))


# ======================================================================
# VERIFY THRESHOLD FILE STRUCTURE
# ======================================================================

required_threshold_columns = [
    "threshold_type",
    "percentile",
    "threshold_mg_m3",
    "source_dataset",
    "target_variable"
]

missing_threshold_columns = [
    col
    for col in required_threshold_columns
    if col not in threshold_df.columns
]

if missing_threshold_columns:
    raise ValueError(
        "Missing threshold columns: "
        + str(missing_threshold_columns)
    )

print("\n✓ Required threshold columns found.")


# ======================================================================
# EXTRACT THE CORRECT CHL THRESHOLDS
# ======================================================================

low_medium_rows = threshold_df[
    threshold_df["threshold_type"]
    == "low_medium_boundary"
]

medium_high_rows = threshold_df[
    threshold_df["threshold_type"]
    == "medium_high_boundary"
]

if len(low_medium_rows) != 1:
    raise ValueError(
        "Expected exactly one low_medium_boundary row."
    )

if len(medium_high_rows) != 1:
    raise ValueError(
        "Expected exactly one medium_high_boundary row."
    )


LOW_MEDIUM_THRESHOLD = float(
    low_medium_rows.iloc[0]["threshold_mg_m3"]
)

MEDIUM_HIGH_THRESHOLD = float(
    medium_high_rows.iloc[0]["threshold_mg_m3"]
)


# ======================================================================
# LOCKED EXPECTED VALUES
# ======================================================================

EXPECTED_LOW_MEDIUM = 0.212671
EXPECTED_MEDIUM_HIGH = 0.433100


print("\n" + "=" * 70)
print("LOCKED TRAINING-ONLY PRODUCTIVITY THRESHOLDS")
print("=" * 70)

print(
    f"\nLow / Medium boundary : "
    f"{LOW_MEDIUM_THRESHOLD:.6f} mg/m³"
)

print(
    f"Medium / High boundary: "
    f"{MEDIUM_HIGH_THRESHOLD:.6f} mg/m³"
)

print("\nExpected locked values:")

print(
    f"Low / Medium : "
    f"{EXPECTED_LOW_MEDIUM:.6f} mg/m³"
)

print(
    f"Medium / High: "
    f"{EXPECTED_MEDIUM_HIGH:.6f} mg/m³"
)


# ======================================================================
# THRESHOLD SAFETY CHECK
# ======================================================================

if not np.isclose(
    LOW_MEDIUM_THRESHOLD,
    EXPECTED_LOW_MEDIUM,
    atol=1e-6
):
    raise ValueError(
        "Low/Medium threshold does not match "
        "the locked training-derived threshold."
    )

if not np.isclose(
    MEDIUM_HIGH_THRESHOLD,
    EXPECTED_MEDIUM_HIGH,
    atol=1e-6
):
    raise ValueError(
        "Medium/High threshold does not match "
        "the locked training-derived threshold."
    )

if LOW_MEDIUM_THRESHOLD >= MEDIUM_HIGH_THRESHOLD:
    raise ValueError(
        "Productivity thresholds are not correctly ordered."
    )

print("\n✓ Correct CHL threshold values confirmed.")
print("✓ Thresholds originate from training data only.")
print("✓ Thresholds were NOT recalculated.")
print("✓ Thresholds remain locked.")


# ======================================================================
# VERIFY THRESHOLD SOURCE
# ======================================================================

source_values = (
    threshold_df["source_dataset"]
    .astype(str)
    .str.lower()
    .unique()
)

if "training_only" not in source_values:
    raise ValueError(
        "Threshold source is not confirmed as training_only."
    )

target_values = (
    threshold_df["target_variable"]
    .astype(str)
    .unique()
)

if "chlorophyll_a" not in target_values:
    raise ValueError(
        "Threshold target is not chlorophyll_a."
    )

print("✓ Threshold source verified as training_only.")
print("✓ Threshold target verified as chlorophyll_a.")


# ======================================================================
# LOAD SAVED TUNED XGBOOST TEST PREDICTIONS
# ======================================================================

print("\n" + "=" * 70)
print("LOADING EXISTING TUNED XGBOOST TEST PREDICTIONS")
print("=" * 70)

predictions_df = pd.read_csv(
    TEST_PREDICTIONS_FILE
)

print(
    f"\nPrediction dataset shape: "
    f"{predictions_df.shape}"
)

print("\nColumns:")
print(predictions_df.columns.tolist())


# ======================================================================
# REQUIRED PREDICTION COLUMN CHECK
# ======================================================================

required_prediction_columns = [
    "date",
    "latitude",
    "longitude",
    "actual_chlorophyll_a",
    "predicted_chlorophyll_a"
]

missing_prediction_columns = [
    col
    for col in required_prediction_columns
    if col not in predictions_df.columns
]

if missing_prediction_columns:
    raise ValueError(
        "Missing prediction columns: "
        + str(missing_prediction_columns)
    )

print("\n✓ Required actual and predicted CHL columns found.")


# ======================================================================
# EXPECTED TEST OBSERVATIONS
# ======================================================================

EXPECTED_TEST_ROWS = 157998

if len(predictions_df) != EXPECTED_TEST_ROWS:
    raise ValueError(
        f"Unexpected number of test prediction rows. "
        f"Expected {EXPECTED_TEST_ROWS}, "
        f"found {len(predictions_df)}."
    )

print(
    f"✓ Test prediction row count confirmed: "
    f"{EXPECTED_TEST_ROWS:,}"
)


# ======================================================================
# CHL SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("CHL PREDICTION SAFETY CHECK")
print("=" * 70)

actual_chl = pd.to_numeric(
    predictions_df["actual_chlorophyll_a"],
    errors="coerce"
)

predicted_chl = pd.to_numeric(
    predictions_df["predicted_chlorophyll_a"],
    errors="coerce"
)

actual_nan = actual_chl.isna().sum()
predicted_nan = predicted_chl.isna().sum()

actual_inf = np.isinf(actual_chl).sum()
predicted_inf = np.isinf(predicted_chl).sum()

print(f"\nActual CHL NaN        : {actual_nan}")
print(f"Predicted CHL NaN    : {predicted_nan}")
print(f"Actual CHL infinite  : {actual_inf}")
print(f"Predicted CHL infinite: {predicted_inf}")

if actual_nan > 0:
    raise ValueError("Actual CHL contains NaN values.")

if predicted_nan > 0:
    raise ValueError("Predicted CHL contains NaN values.")

if actual_inf > 0:
    raise ValueError("Actual CHL contains infinite values.")

if predicted_inf > 0:
    raise ValueError("Predicted CHL contains infinite values.")

print("\n✓ CHL prediction values passed all safety checks.")


# ======================================================================
# PRODUCTIVITY CLASSIFICATION
# ======================================================================

print("\n" + "=" * 70)
print("APPLYING PRODUCTIVITY CLASSIFICATION")
print("=" * 70)

print("""
Classification rules:

LOW:
    CHL < 0.212671 mg/m³

MEDIUM:
    0.212671 <= CHL <= 0.433100 mg/m³

HIGH:
    CHL > 0.433100 mg/m³
""")


def classify_productivity(chl):
    if chl < LOW_MEDIUM_THRESHOLD:
        return "Low"

    elif chl <= MEDIUM_HIGH_THRESHOLD:
        return "Medium"

    else:
        return "High"


predictions_df["actual_productivity_class"] = (
    actual_chl.apply(classify_productivity)
)

predictions_df["predicted_productivity_class"] = (
    predicted_chl.apply(classify_productivity)
)

print("✓ Actual productivity classes assigned.")
print("✓ Predicted productivity classes assigned.")


# ======================================================================
# CLASS ORDER
# ======================================================================

CLASS_ORDER = [
    "Low",
    "Medium",
    "High"
]


# ======================================================================
# ACTUAL PRODUCTIVITY DISTRIBUTION
# ======================================================================

print("\n" + "=" * 70)
print("ACTUAL PRODUCTIVITY CLASS DISTRIBUTION")
print("=" * 70)

actual_counts = (
    predictions_df["actual_productivity_class"]
    .value_counts()
    .reindex(CLASS_ORDER, fill_value=0)
)

actual_distribution = pd.DataFrame({
    "productivity_class": CLASS_ORDER,
    "observations": actual_counts.values
})

actual_distribution["percentage"] = (
    actual_distribution["observations"]
    / len(predictions_df)
    * 100
)

print(
    actual_distribution.to_string(index=False)
)


# ======================================================================
# PREDICTED PRODUCTIVITY DISTRIBUTION
# ======================================================================

print("\n" + "=" * 70)
print("PREDICTED PRODUCTIVITY CLASS DISTRIBUTION")
print("=" * 70)

predicted_counts = (
    predictions_df["predicted_productivity_class"]
    .value_counts()
    .reindex(CLASS_ORDER, fill_value=0)
)

predicted_distribution = pd.DataFrame({
    "productivity_class": CLASS_ORDER,
    "observations": predicted_counts.values
})

predicted_distribution["percentage"] = (
    predicted_distribution["observations"]
    / len(predictions_df)
    * 100
)

print(
    predicted_distribution.to_string(index=False)
)


# ======================================================================
# CONFUSION MATRIX
# ======================================================================

print("\n" + "=" * 70)
print("PRODUCTIVITY CONFUSION MATRIX")
print("=" * 70)

cm = confusion_matrix(
    predictions_df["actual_productivity_class"],
    predictions_df["predicted_productivity_class"],
    labels=CLASS_ORDER
)

confusion_df = pd.DataFrame(
    cm,
    index=[
        "Actual Low",
        "Actual Medium",
        "Actual High"
    ],
    columns=[
        "Predicted Low",
        "Predicted Medium",
        "Predicted High"
    ]
)

print()
print(confusion_df.to_string())


# ======================================================================
# CLASSIFICATION METRICS
# ======================================================================

print("\n" + "=" * 70)
print("OVERALL PRODUCTIVITY CLASSIFICATION PERFORMANCE")
print("=" * 70)

y_true = predictions_df[
    "actual_productivity_class"
]

y_pred = predictions_df[
    "predicted_productivity_class"
]


accuracy = accuracy_score(
    y_true,
    y_pred
)

macro_precision = precision_score(
    y_true,
    y_pred,
    labels=CLASS_ORDER,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_true,
    y_pred,
    labels=CLASS_ORDER,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=CLASS_ORDER,
    average="macro",
    zero_division=0
)

weighted_precision = precision_score(
    y_true,
    y_pred,
    labels=CLASS_ORDER,
    average="weighted",
    zero_division=0
)

weighted_recall = recall_score(
    y_true,
    y_pred,
    labels=CLASS_ORDER,
    average="weighted",
    zero_division=0
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    labels=CLASS_ORDER,
    average="weighted",
    zero_division=0
)


print(f"\nAccuracy           : {accuracy:.6f}")
print(f"Macro Precision    : {macro_precision:.6f}")
print(f"Macro Recall       : {macro_recall:.6f}")
print(f"Macro F1           : {macro_f1:.6f}")
print(f"Weighted Precision : {weighted_precision:.6f}")
print(f"Weighted Recall    : {weighted_recall:.6f}")
print(f"Weighted F1        : {weighted_f1:.6f}")


# ======================================================================
# CLASS-WISE PERFORMANCE
# ======================================================================

print("\n" + "=" * 70)
print("CLASS-WISE PRODUCTIVITY PERFORMANCE")
print("=" * 70)

class_precision = precision_score(
    y_true,
    y_pred,
    labels=CLASS_ORDER,
    average=None,
    zero_division=0
)

class_recall = recall_score(
    y_true,
    y_pred,
    labels=CLASS_ORDER,
    average=None,
    zero_division=0
)

class_f1 = f1_score(
    y_true,
    y_pred,
    labels=CLASS_ORDER,
    average=None,
    zero_division=0
)

class_support = (
    y_true
    .value_counts()
    .reindex(CLASS_ORDER, fill_value=0)
    .values
)

classwise_df = pd.DataFrame({
    "productivity_class": CLASS_ORDER,
    "precision": class_precision,
    "recall": class_recall,
    "f1_score": class_f1,
    "support": class_support
})

print(
    classwise_df.to_string(index=False)
)


# ======================================================================
# CLASSIFICATION AGREEMENT
# ======================================================================

correct_classifications = (
    y_true == y_pred
).sum()

total_observations = len(predictions_df)

agreement_percentage = (
    correct_classifications
    / total_observations
    * 100
)

print("\n" + "=" * 70)
print("PRODUCTIVITY CLASSIFICATION AGREEMENT")
print("=" * 70)

print(
    f"\nCorrect classifications: "
    f"{correct_classifications:,}"
)

print(
    f"Total observations: "
    f"{total_observations:,}"
)

print(
    f"Agreement: "
    f"{agreement_percentage:.4f}%"
)


# ======================================================================
# SAVE CLASSIFIED TEST PREDICTIONS
# ======================================================================

classified_predictions_file = os.path.join(
    OUTPUT_DIR,
    "xgboost_final_tuned_test_productivity_predictions.csv"
)

predictions_df.to_csv(
    classified_predictions_file,
    index=False
)

print("\n" + "=" * 70)
print("CLASSIFIED TEST PREDICTIONS SAVED")
print("=" * 70)

print(classified_predictions_file)


# ======================================================================
# SAVE CONFUSION MATRIX
# ======================================================================

confusion_matrix_file = os.path.join(
    OUTPUT_DIR,
    "final_tuned_productivity_confusion_matrix.csv"
)

confusion_df.to_csv(
    confusion_matrix_file
)

print(confusion_matrix_file)


# ======================================================================
# SAVE CLASS-WISE METRICS
# ======================================================================

classwise_metrics_file = os.path.join(
    OUTPUT_DIR,
    "final_tuned_productivity_classwise_metrics.csv"
)

classwise_df.to_csv(
    classwise_metrics_file,
    index=False
)

print(classwise_metrics_file)


# ======================================================================
# SAVE OVERALL METRICS
# ======================================================================

overall_metrics_df = pd.DataFrame({
    "metric": [
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_precision",
        "weighted_recall",
        "weighted_f1",
        "correct_classifications",
        "total_observations",
        "classification_agreement_percent"
    ],
    "value": [
        accuracy,
        macro_precision,
        macro_recall,
        macro_f1,
        weighted_precision,
        weighted_recall,
        weighted_f1,
        correct_classifications,
        total_observations,
        agreement_percentage
    ]
})

overall_metrics_file = os.path.join(
    OUTPUT_DIR,
    "final_tuned_productivity_overall_metrics.csv"
)

overall_metrics_df.to_csv(
    overall_metrics_file,
    index=False
)

print(overall_metrics_file)


# ======================================================================
# SAVE THRESHOLD RECORD
# ======================================================================

threshold_record = {
    "threshold_source": "training_only",
    "target_variable": "chlorophyll_a",

    "low_medium_boundary_mg_m3":
        LOW_MEDIUM_THRESHOLD,

    "medium_high_boundary_mg_m3":
        MEDIUM_HIGH_THRESHOLD,

    "thresholds_recalculated": False,
    "validation_data_loaded": False,
    "raw_test_dataset_loaded": False,
    "test_predictions_regenerated": False
}

threshold_record_file = os.path.join(
    OUTPUT_DIR,
    "final_tuned_productivity_threshold_record.json"
)

with open(threshold_record_file, "w") as f:
    json.dump(
        threshold_record,
        f,
        indent=4
    )

print(threshold_record_file)


# ======================================================================
# SAVE COMPLETE SUMMARY
# ======================================================================

summary = {
    "step": "31.9P-16R",
    "model": "Final Tuned XGBoost",
    "classification_type":
        "CHL-based productivity classification",

    "threshold_source": "training_only",

    "low_medium_threshold_mg_m3":
        float(LOW_MEDIUM_THRESHOLD),

    "medium_high_threshold_mg_m3":
        float(MEDIUM_HIGH_THRESHOLD),

    "test_predictions_source":
        "already_generated_final_tuned_xgboost_predictions",

    "raw_test_dataset_loaded": False,

    "test_predictions_regenerated": False,

    "xgboost_retrained": False,

    "hyperparameters_changed": False,

    "accuracy": float(accuracy),

    "macro_precision": float(macro_precision),

    "macro_recall": float(macro_recall),

    "macro_f1": float(macro_f1),

    "weighted_precision": float(weighted_precision),

    "weighted_recall": float(weighted_recall),

    "weighted_f1": float(weighted_f1),

    "correct_classifications":
        int(correct_classifications),

    "total_observations":
        int(total_observations),

    "classification_agreement_percent":
        float(agreement_percentage)
}

summary_file = os.path.join(
    OUTPUT_DIR,
    "final_tuned_productivity_classification_summary.json"
)

with open(summary_file, "w") as f:
    json.dump(
        summary,
        f,
        indent=4
    )

print(summary_file)


# ======================================================================
# FINAL SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16R SAFETY CHECK")
print("=" * 70)

print("\nTraining-derived thresholds used : TRUE")
print("Thresholds recalculated           : FALSE")
print("Validation data loaded            : FALSE")
print("Raw test dataset loaded           : FALSE")
print("Existing tuned test predictions used: TRUE")
print("Test predictions regenerated      : FALSE")
print("XGBoost retraining                : FALSE")
print("XGBoost modification              : FALSE")
print("Hyperparameters changed           : FALSE")

print("\n✓ Correct training-only thresholds applied.")
print("✓ Existing final tuned XGBoost test predictions used.")
print("✓ No test predictions were regenerated.")
print("✓ No XGBoost model was retrained.")
print("✓ No hyperparameters were changed.")
print("✓ Confusion matrix calculated.")
print("✓ Overall classification metrics calculated.")
print("✓ Class-wise metrics calculated.")
print("✓ Productivity classification outputs saved.")


# ======================================================================
# FINAL SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("STEP 31.9P-16R COMPLETE")
print("=" * 70)

print("\nFINAL TUNED XGBOOST PRODUCTIVITY CLASSIFICATION")

print("\nTraining-derived boundaries:")

print(
    f"\nLOW:\n"
    f"    CHL < {LOW_MEDIUM_THRESHOLD:.6f} mg/m³"
)

print(
    f"\nMEDIUM:\n"
    f"    {LOW_MEDIUM_THRESHOLD:.6f} <= CHL <= "
    f"{MEDIUM_HIGH_THRESHOLD:.6f} mg/m³"
)

print(
    f"\nHIGH:\n"
    f"    CHL > {MEDIUM_HIGH_THRESHOLD:.6f} mg/m³"
)

print(f"\nAccuracy    : {accuracy:.6f}")
print(f"Macro F1    : {macro_f1:.6f}")
print(f"Weighted F1 : {weighted_f1:.6f}")

print("\nNEXT:")
print("31.9P-16S — SHAP Explainability")

print("=" * 70)

STEP 31.9P-16R
FINAL TUNED XGBOOST PRODUCTIVITY CLASSIFICATION

PROJECT PATHS

Project:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Final tuned XGBoost directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate

Tuned XGBoost test predictions:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate/final_test_evaluation/xgboost_final_tuned_test_predictions.csv

Training-only productivity thresholds:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/18_productivity_classification/productivity_thresholds_training_only.csv

Output directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate/productivity_classification

REQUIRED FILE CHECK

Tuned XGBoost test predictions: True
Training-only threshold file: True

✓ All required files found.

METHODOLOGICAL SAFETY

Trai